In [ ]:
# Install everything the notebook needs (harmless if already installed).
# Note for Python 3.14: torch >= 2.9 ships wheels for it. If pip cannot resolve a
# wheel for your interpreter, create a 3.12/3.13 venv instead:
#     python3.12 -m venv .venv && source .venv/bin/activate
#     pip install jupyter numpy matplotlib torch gymnasium
%pip install numpy matplotlib torch gymnasium

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
plt.rcParams["animation.html"] = "jshtml"

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)

def smooth(x, w=10):
    x = np.asarray(x, dtype=float)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w) / w, mode="valid")

def greedy_action(Q, s):
    q = Q[s]
    return int(np.random.choice(np.flatnonzero(q == q.max())))

def compute_returns(rewards, gamma):
    # G_t = sum_{k>=0} gamma^k r_{t+k}, accumulated backwards
    G = np.zeros_like(rewards, dtype=float)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running
        G[t] = running
    return G

def plot_heatmap(ax, M, title, gw=None, fmt="{:.2f}"):
    im = ax.imshow(M, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, fmt.format(M[i, j]), ha="center", va="center", fontsize=9)
    if gw is not None:
        ax.text(gw.goal[1], gw.goal[0], "★", fontsize=22, ha="center", va="center", color="gold")
        ax.text(gw.trap[1], gw.trap[0], "☠", fontsize=20, ha="center", va="center", color="red")

def plot_policy_grid(ax, gw, Q, title):
    V = Q.max(axis=1)
    plot_heatmap(ax, V.reshape(gw.size, gw.size), title, gw=gw)
    for s in range(gw.n_states):
        r, c = divmod(s, gw.size)
        if (r, c) in (gw.goal, gw.trap):
            continue
        dr, dc = gw.ACTIONS[int(np.argmax(Q[s]))]
        ax.arrow(c, r, 0.30 * dc, 0.30 * dr, head_width=0.16, head_length=0.14,
                 color="white", lw=2, length_includes_head=True)

class Gridworld:
    ACTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # up, right, down, left

    def __init__(self, size=5, goal=(4, 4), trap=(2, 2), r_step=-1.0, r_goal=10.0, r_trap=-5.0):
        self.size = size
        self.goal, self.trap = goal, trap
        self.r_step, self.r_goal, self.r_trap = r_step, r_goal, r_trap
        self.n_states = size * size
        self.n_actions = 4
        self.start = 0
        self.s = self.start
        # exact transition model P[s, a, s'] and expected reward R[s, a]
        # (known only because we built the world — the agent never sees these)
        self.P = np.zeros((self.n_states, self.n_actions, self.n_states))
        self.R = np.zeros((self.n_states, self.n_actions))
        for s in range(self.n_states):
            r, c = divmod(s, size)
            if (r, c) in (goal, trap):
                continue  # terminal: no outgoing transitions
            for a, (dr, dc) in enumerate(self.ACTIONS):
                nr, nc = r + dr, c + dc
                if 0 <= nr < size and 0 <= nc < size:
                    ns = nr * size + nc
                else:
                    ns = s  # wall: stay put
                self.P[s, a, ns] = 1.0
                if (nr, nc) == goal:
                    self.R[s, a] = r_goal
                elif (nr, nc) == trap:
                    self.R[s, a] = r_trap
                else:
                    self.R[s, a] = r_step

    def reset(self):
        self.s = self.start
        return self.s

    def step(self, a):
        if divmod(self.s, self.size) in (self.goal, self.trap):
            return self.s, 0.0, True
        s_next = int(np.flatnonzero(self.P[self.s, a])[0])
        r = self.R[self.s, a]
        self.s = s_next
        done = divmod(s_next, self.size) in (self.goal, self.trap)
        return s_next, r, done

In [ ]:
# Quick sanity check of the three environments we will use (gymnasium 1.x API)
for env_id in ["CartPole-v1", "CliffWalking-v1", "Acrobot-v1"]:
    e = gym.make(env_id)
    print(f"{env_id:>15}  obs: {e.observation_space}   action: {e.action_space}")
    e.close()

env = gym.make("CartPole-v1")
obs, info = env.reset(seed=SEED)
for _ in range(3):
    obs, r, terminated, truncated, info = env.step(env.action_space.sample())
print("step -> obs:", obs, " reward:", r, " terminated:", terminated, " truncated:", truncated)
env.close()

## Foundations

- RL is learning by *interacting* — not from a fixed labeled dataset.
- **The loop:** the environment gives the agent a state $s_t$; the agent picks an action $a_t$; the environment responds with a reward $r_{t+1}$ and a new state $s_{t+1}$.
- An interaction is a trajectory $(s_0,a_0,r_0),(s_1,a_1,r_1),\dots$; the **Markov property** says $s_{t+1}$ depends only on $s_t$ (and $a_t$), so the current state is all the agent needs.
- The goal is to maximize *expected future* reward — long-term, not just the next step.
- Three paradigms — value learning (learn a **critic**: $V$, $Q$), policy gradients (learn an **actor**: $\pi$), and actor-critic (both).

In [ ]:
# A real MDP trajectory: (s_t, a_t, r_t) tuples.
# (The Gridworld class lives in the helpers cell above; more on it in the MC section.)
set_seed(SEED)  # reproducible random policy
gw_demo = Gridworld()
s = gw_demo.reset()
traj = []
for _ in range(6):
    a = np.random.choice(gw_demo.n_actions)
    s_next, r, done = gw_demo.step(a)
    traj.append((s, a, r))
    s = s_next
    if done:
        break
pos = divmod(s, gw_demo.size)
end = "goal ★" if pos == gw_demo.goal else "trap ☠" if pos == gw_demo.trap else "still going"
print("trajectory (state, action, reward):")
for i, t in enumerate(traj):
    print(f"{i+1}. {t}")
print("ended in:", end)
print("the Markov property: to pick the next action, the agent only needs s_7 =", s)

In [ ]:
def plot_discounting():
    gammas = [0.0, 0.5, 0.9, 0.99]
    T = 30
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    for g in gammas:
        w = g ** np.arange(T)
        axes[0].plot(w, lw=2, label=rf"$\gamma={g}$")
        axes[1].plot(w.cumsum(), lw=2, label=rf"$\gamma={g}$")
    axes[0].set_title(r"weight of a reward $k$ steps ahead: $\gamma^k$")
    axes[0].set_xlabel("k (steps in the future)")
    axes[0].set_ylabel(r"$\gamma^k$")
    axes[0].grid(alpha=0.3)
    axes[0].legend()
    axes[1].set_title(r"return $G_t=\sum_{k=0}^{29}\gamma^k\cdot 1$ (constant reward 1)")
    axes[1].set_xlabel("steps ahead k")
    axes[1].set_ylabel("cumulative return")
    axes[1].grid(alpha=0.3)
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_discounting()

In [ ]:
def plot_eps_greedy():
    q = np.array([1.0, 3.0, 2.0])   # action a2 currently looks best
    eps_list = [0.0, 0.1, 0.5]
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2))
    for ax, eps in zip(axes, eps_list):
        p = np.full(3, eps / 3)
        p[np.argmax(q)] += 1 - eps
        colors = ["#C44E52" if i == np.argmax(q) else "#4C72B0" for i in range(3)]
        ax.bar(range(3), p, color=colors)
        ax.set_title(rf"$\varepsilon={eps}$")
        ax.set_xticks(range(3), ["a1 (Q=1)", "a2 (Q=3)", "a3 (Q=2)"])
        ax.set_ylim(0, 1.05)
        ax.set_ylabel(r"$p(a\mid s)$")
    fig.suptitle("ε-greedy action selection: exploit with prob 1−ε, explore uniformly with prob ε")
    plt.tight_layout()
    plt.show()

plot_eps_greedy()

## Monte Carlo value learning

- The **return** is the expected discounted future reward
  $\mathbb E_\pi\left[\sum_{k=0}^{\infty}\gamma^k r_{t+k}\right]$.
- $V(s)=\mathbb E_\pi\left[\sum_{k=0}^{\infty}\gamma^k r_{t+k}\mid S=s\right]$ — *"how good is the place I'm at?"*
- $Q(s,a)=\mathbb E_\pi\left[\sum_{k=0}^{\infty}\gamma^k r_{t+k}\mid S=s,A=a\right]$ — *"how good is this action?"*
- Monte Carlo learns from **complete episodes**. After each episode, every visited $(s_t,a_t)$ is nudged toward the return actually observed from there:

$$V(s_t)\leftarrow V(s_t)+\alpha\Big[\sum_{k=0}^{T-t}\gamma^k r_{t+k}-V(s_t)\Big]\qquad
Q(s_t,a_t)\leftarrow Q(s_t,a_t)+\alpha\Big[\sum_{k=0}^{T-t}\gamma^k r_{t+k}-Q(s_t,a_t)\Big]$$

- Unbiased but high-variance targets, and you must wait for the episode to end.

Our environment is a 5×5 gridworld: every step costs −1, the ★ goal pays +10, the ☠ trap costs −5, walls block movement, and episodes start in the top-left corner.

In [ ]:
def solve_true_v(gw, gamma=0.9, policy=None):
    # Closed-form V for a known policy, from the Bellman equation (I - gamma P^pi) V = R^pi.
    # Possible only because WE built the world and know its exact dynamics.
    # The agent never gets to do this — it must learn from episodes.
    if policy is None:
        policy = np.ones((gw.n_states, gw.n_actions)) / gw.n_actions
    P_pi = np.einsum("sa,san->sn", policy, gw.P)
    R_pi = np.einsum("sa,sa->s", policy, gw.R)
    return np.linalg.solve(np.eye(gw.n_states) - gamma * P_pi, R_pi)

gw = Gridworld()
V_true = solve_true_v(gw, gamma=0.9)

fig, ax = plt.subplots(figsize=(4.6, 4))
plot_heatmap(ax, V_true.reshape(gw.size, gw.size),
             "True V under a uniform random policy\n(solved from the Bellman equations)", gw=gw)
plt.show()

In [ ]:
def mc_predict_v(gw, gamma=0.9, alpha=0.05, n_episodes=2000, seed=SEED):
    # Monte Carlo prediction: V(s_t) <- V(s_t) + alpha [ sum_{k=0}^{T-t} gamma^k r_{t+k} - V(s_t) ]
    # every-visit Monte Carlo prediction under a uniform random policy.
    set_seed(seed)
    V = np.zeros(gw.n_states)
    V_true = solve_true_v(gw, gamma=gamma)
    rmse_log = []
    for ep in range(n_episodes):
        s = gw.reset()
        traj = []
        while True:
            a = np.random.choice(gw.n_actions)
            s_next, r, done = gw.step(a)
            traj.append((s, a, r))
            s = s_next
            if done:
                break
        G = 0.0
        for (s_, a_, r_) in reversed(traj):
            G = r_ + gamma * G
            V[s_] += alpha * (G - V[s_])
        if (ep + 1) % 100 == 0:
            rmse_log.append(np.sqrt(np.mean((V - V_true) ** 2)))
    return V, V_true, rmse_log

V_est, V_true, rmse_log = mc_predict_v(gw, gamma=0.9, alpha=0.05, n_episodes=2000)
print("final RMSE vs true V:", round(rmse_log[-1], 4))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
plot_heatmap(axes[0], V_true.reshape(gw.size, gw.size), "True V (Bellman solution)", gw=gw)
plot_heatmap(axes[1], V_est.reshape(gw.size, gw.size), "MC estimate after 2000 episodes", gw=gw)
axes[2].plot(np.arange(100, 2001, 100), rmse_log, "o-", color="#C44E52")
axes[2].set_xlabel("episode")
axes[2].set_ylabel("RMSE vs true V")
axes[2].set_title("Monte Carlo converges to the true value")
axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Reading the heatmaps:** the trap gets a negative value (a random walk ends there about half the time), the goal and its neighborhood are positive, and values fade with distance — exactly the intuition $V(s)$ = *"how good is the place I'm at?"*.

Now the full agent. The ε-greedy rule gives the **behavior policy**: with probability $\varepsilon$ act randomly, otherwise act greedily on $Q$. Then the Monte Carlo update learns $Q$ from the episodes this behavior produces.

In [ ]:
def mc_control_q(gw, gamma=0.9, alpha=0.1, eps=0.1, n_episodes=2000, max_steps=200, seed=SEED):
    # Monte Carlo control: the same update for Q, acting epsilon-greedy.
    set_seed(seed)
    Q = np.zeros((gw.n_states, gw.n_actions))
    ep_len_log = []

    #over multiple episodes
    for ep in range(n_episodes):
        s = gw.reset()
        traj = []
        steps = 0

        # gather trajectory using epsilon greedy policy
        while steps < max_steps:
            random_chance = np.random.rand()
            if random_chance < eps: # policy will take a random action
                a = np.random.choice(gw.n_actions)
            else:
                a = greedy_action(Q, s) # policy will take a greedy action

            s_next, r, done = gw.step(a) # get s_t+1, r_t, and done flag from environment

            #TODO: append the (s_t, a_t, r_t) tuple to trajectory

            s = s_next #update the state
            steps += 1
            if done:
                break # if terminal state reached

        # calculate returns from back of the trajectory
        # G_T   = r_T 
        # G_T-1 = r_T-1 + gamma * (G_T)   = r_T-1 + gamma * r_T 
        # G_T-2 = r_T-2 + gamma * (G_T-1) = r_T-2 + gamma * r_T-1 + gamma^2 * r_T
        # and so on until G_1
        G = 0.0
        for (s_, a_, r_) in reversed(traj): # from timestep T down to timestep 1
            # TODO: accumulate return for that step

            # TODO: update the Q table according to the MC update rule
            
            pass # TODO: REMOVE THIS AFTER COMPLETING THE ABOVE TWO STEPS

        ep_len_log.append(steps)

    return Q, ep_len_log

Q_mc, ep_len_log = mc_control_q(gw, gamma=0.9, alpha=0.1, eps=0.1, n_episodes=2000)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
plot_policy_grid(axes[0], gw, Q_mc, "MC control: V = max_a Q(s,a)\narrows = greedy policy")
axes[1].plot(smooth(ep_len_log, 20), color="#4C72B0", lw=2)
axes[1].set_xlabel("episode")
axes[1].set_ylabel("episode length (smoothed)")
axes[1].set_title("Episodes get shorter as the agent learns the way to the goal")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

The arrows point toward the goal and away from the trap — the greedy policy induced from $Q$ **is** a plan.

**But:** Monte Carlo had to wait for the episode to end before it could learn anything, and its targets swing wildly between episodes. What if we could update *every single step*, using our own current prediction of the future as part of the target? That is temporal-difference learning.

## Temporal-difference learning

- The **TD error** $\delta_t = \big(r_t+\gamma V(s_{t+1})\big)-V(s_t)$ — observed reward plus our *predicted* future, minus the current prediction. *"How surprised should I be?"*
- Update immediately, every step: $Q(s_t,a_t)\leftarrow Q(s_t,a_t)+\alpha\,\delta_t$. No waiting for the episode to end.
- For action values, $\delta_t = r_t+\gamma Q(s_{t+1},a')-Q(s_t,a_t)$ — and now the key question: **which $a'$?**
- **SARSA:** the *actual* next action, $a'=a_{t+1}$ (on-policy; named for the tuple $s_t,a_t,r_t,s_{t+1},a_{t+1}$).
- **Expected SARSA:** the *expectation* over next actions, $\mathbb E_{a\in A}[Q(s_{t+1},a)]$ — robust to one unlucky exploratory action.
- **Q-learning:** the *optimistic* max, $\max_{a'}Q(s_{t+1},a')$ (off-policy; learns the optimal $Q^*$).

The maze: **CliffWalking** — start at **S**, reach **G**, but the brown cells are a cliff: stepping into one costs −100 and throws the agent back to the start. The shortest path grazes the cliff; the safe path takes the long way around.

In [ ]:
def eval_greedy(env, agent, seed, gamma=1.0, max_steps=200):
    # a clean evaluation of the learned greedy policy (epsilon = 0),
    # returning the same discounted sum the agent is learning against
    set_seed(seed)
    s, _ = env.reset(seed=seed)
    R, done = 0, False
    while not done:
        a = greedy_action(agent.Q, s)
        s, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        R = r + gamma * R
    return R


class TabularTD:
    # One class, three update rules.
    def __init__(self, n_states, n_actions, mode, alpha=0.5, gamma=1.0, eps=0.1):
        self.mode = mode
        self.Q = np.zeros((n_states, n_actions))
        self.alpha, self.gamma, self.eps = alpha, gamma, eps
        self.n_actions = n_actions

    def act(self, s, greedy=False):
        if not greedy and np.random.rand() < self.eps:
            return np.random.choice(self.n_actions)
        return greedy_action(self.Q, s)

    def update(self, s, a, r, s_next, a_next, done):
        if done:
            target = r
        elif self.mode == "sarsa":
            # SARSA: use the ACTUAL next action from the trajectory
            
            #TODO: add in the SARSA update target (see expected sarsa below for help)
            target = None
        elif self.mode == "expected_sarsa":
            # Expected SARSA: average over next actions under the current eps-greedy policy
            qs = self.Q[s_next]
            target = r + self.gamma * (
                (1 - self.eps) * qs.max() + (self.eps / self.n_actions) * qs.sum())
        else:  # qlearning
            # Q-learning: assume the best next action (off-policy)
            
            # TODO: add in the Q learning update target
            # HINT: .max() function takes the max value from a numpy array
            target = None

        self.Q[s, a] += self.alpha * (target - self.Q[s, a]) #update the Q table


def run_td(env_id, mode, n_episodes=500, alpha=0.5, gamma=1.0, eps=0.1,
           seed=SEED, max_steps=200, eval_every=25):
    set_seed(seed)
    env = gym.make(env_id)
    env = gym.wrappers.TimeLimit(env, max_episode_steps=max_steps)
    agent = TabularTD(env.observation_space.n, env.action_space.n, mode, alpha, gamma, eps)
    returns, lengths, eval_returns = [], [], []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=seed + ep)
        a = agent.act(s)
        R, steps, done = 0, 0, False
        while not done:
            s_next, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated

            if done:
                agent.update(s, a, r, s_next, None, True)
            else:
                a_next = agent.act(s_next)
                agent.update(s, a, r, s_next, a_next, False)
                a = a_next

            s = s_next
            R = r + gamma * R
            steps += 1
        returns.append(R)
        lengths.append(steps)
        if (ep + 1) % eval_every == 0:
            eval_returns.append(eval_greedy(env, agent, seed + 10000 + ep, gamma=gamma))
    env.close()
    return agent, returns, lengths, eval_returns

In [ ]:
RESULTS = {"td": {}}
td_agents = {}
for mode in ["sarsa", "expected_sarsa", "qlearning"]:
    agent, returns, lengths, eval_returns = run_td("CliffWalking-v1", mode, n_episodes=500)
    td_agents[mode] = agent
    RESULTS["td"][mode] = (returns, lengths, eval_returns)
    print(f"{mode:>15}: final greedy return = {eval_returns[-1]:.1f}   "
          f"final smoothed behavior return = {smooth(returns, 20)[-1]:.1f}")

colors = {"sarsa": "#4C72B0", "expected_sarsa": "#DD8452", "qlearning": "#55A868"}
fig, axes = plt.subplots(1, 3, figsize=(15.5, 3.8))
for mode, (returns, lengths, eval_returns) in RESULTS["td"].items():
    axes[0].plot(smooth(returns, 20), lw=2, color=colors[mode], label=mode)
    axes[1].plot(np.arange(25, 501, 25), eval_returns, "o-", ms=4, lw=2,
                 color=colors[mode], label=mode)
    axes[2].plot(smooth(lengths, 20), lw=2, color=colors[mode], label=mode)
axes[0].set_title("behavior return (ε-greedy, ε=0.1) —\nQ-learning's exploration keeps falling in the cliff")
axes[0].set_xlabel("episode")
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[1].set_title("greedy-policy return (eval every 25 episodes) —\nQ-learning ≈ −13 (optimal), SARSA ≈ −17 (safe)")
axes[1].set_xlabel("episode")
axes[1].grid(alpha=0.3)
axes[1].legend()
axes[2].set_title("episode length — SARSA takes the long safe way")
axes[2].set_xlabel("episode")
axes[2].grid(alpha=0.3)
axes[2].legend()
plt.tight_layout()
plt.show()

In [ ]:
def plot_cliff_q(ax, agent, title):
    V = agent.Q.max(axis=1).reshape(4, 12)
    im = ax.imshow(V, cmap="viridis")
    # draw the world on top: cliff cells along the bottom row, goal bottom-right
    for c in range(1, 11):
        ax.add_patch(plt.Rectangle((c - 0.5, 0), 1, 1, color="#8B4513", alpha=0.85))
    ax.add_patch(plt.Rectangle((11, 0), 1, 1, color="green", alpha=0.5))
    ax.text(0.5, 0.5, "S", ha="center", va="center", fontweight="bold", fontsize=12)
    ax.text(11.5, 0.5, "G", ha="center", va="center", fontweight="bold", fontsize=12)
    for r in range(4):
        for c in range(12):
            if r == 3 and 0 < c < 11:
                continue  # cliff cells are never occupied
            ax.text(c + 0.5, 3.5 - r, f"{V[r, c]:.0f}", ha="center", va="center", fontsize=6)
    ax.set_ylim(-0.5, 3.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_cliff_q(axes[0], td_agents["sarsa"], "SARSA")
plot_cliff_q(axes[1], td_agents["expected_sarsa"], "Expected SARSA")
plot_cliff_q(axes[2], td_agents["qlearning"], "Q-learning")
fig.suptitle(r"learned $V(s)=\max_a Q(s,a)$ on the CliffWalking maze")
plt.tight_layout()
plt.show()

In [ ]:
def animate_cliff(agent, title, seed=SEED, max_steps=80):
    set_seed(seed)
    env = gym.make("CliffWalking-v1")
    env = gym.wrappers.TimeLimit(env, max_episode_steps=max_steps)
    s, _ = env.reset(seed=seed)
    path, done, R, steps = [s], False, 0, 0
    while not done:
        a = greedy_action(agent.Q, s)
        s, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        path.append(s)
        R += r
        steps += 1
    env.close()

    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.set_xlim(-0.5, 11.5)
    ax.set_ylim(-0.5, 3.5)
    ax.set_aspect("equal")
    for c in range(1, 11):
        ax.add_patch(plt.Rectangle((c - 0.5, 0), 1, 1, color="#8B4513", alpha=0.85))
    ax.add_patch(plt.Rectangle((11, 0), 1, 1, color="green", alpha=0.5))
    ax.text(-0.9, 0.5, "START", ha="right", va="center", fontsize=9)
    ax.text(12.4, 0.5, "GOAL", ha="left", va="center", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    pts = [(s_ % 12 + 0.5, 3.5 - s_ // 12) for s_ in path]
    line, = ax.plot([], [], "o-", color="#C44E52", lw=2.5, ms=8)
    ax.set_title(f"{title}: greedy path after training (return = {R})")

    def update(i):
        j = min(i, len(pts) - 1)
        line.set_data([p[0] for p in pts[:j + 1]], [p[1] for p in pts[:j + 1]])
        return (line,)

    ani = animation.FuncAnimation(fig, update, frames=len(pts) + 6, interval=350, blit=True)
    html = ani.to_jshtml()
    plt.close(fig)
    return html

print("SARSA — the safe path along the top edge:")
display(HTML(animate_cliff(td_agents["sarsa"], "SARSA")))
print("Q-learning — the risky optimal path right along the cliff:")
display(HTML(animate_cliff(td_agents["qlearning"], "Q-learning")))

**Why the difference?** SARSA is **on-policy**: it evaluates the actions its own ε-greedy policy actually takes — and an occasional random step near the cliff edge is catastrophic, so it learns to stay far away. Q-learning is **off-policy**: it evaluates the *best* next action regardless of how it behaves, so it can keep hugging the optimal-but-dangerous edge while exploring.

Expected SARSA sits between them: it averages over the exploration noise, so its value estimates resemble Q-learning's while its behavior stays a bit more cautious.

## Deep Q-Networks

- A table has one entry per $(s,a)$ — impossible for large state spaces. Replace the table with a neural network.
- Input = state, output = one $Q(s,a)$ per action — a single network covers the whole state space.
- Training error $= r_t+\gamma\max_{a'}Q^{-}(s_{t+1},a')-Q_\theta(s_t,a_t)$, where $Q^{-}$ is a **target network**: a frozen copy of $Q_\theta$ that lags behind by several updates, so the target stops moving while we train toward it.
- Batch loss with a stop-gradient on the target (`.detach()` in PyTorch):

$$\mathcal L(\theta)=\frac{1}{M}\sum_{i=1}^{M}\Big(r_i+\gamma\,\operatorname{sg}\big[\max_{a'}Q^{-}(s_{i+1},a')\big]-Q_\theta(s_i,a_i)\Big)^2$$

We also use an **experience replay buffer** — a standard DQN ingredient: store past transitions and train on shuffled mini-batches to break their correlation.

In [ ]:
class QNetwork(nn.Module):
    # state in, one Q-value per action out
    def __init__(self, obs_dim, n_actions, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, s):
        return self.net(s)

# similar to a dataloader
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.capacity = capacity
        self.buf = []
        self.pos = 0

    def push(self, s, a, r, s_next, terminated):
        item = (np.asarray(s, dtype=np.float32), a, r,
                np.asarray(s_next, dtype=np.float32), float(terminated))
        if len(self.buf) < self.capacity:
            self.buf.append(item)
        else:
            self.buf[self.pos % self.capacity] = item
        self.pos += 1

    def sample(self, batch):
        idx = np.random.randint(0, len(self.buf), batch)
        S = np.stack([self.buf[i][0] for i in idx])
        A = np.array([self.buf[i][1] for i in idx], dtype=np.int64)
        R = np.array([self.buf[i][2] for i in idx], dtype=np.float32)
        S2 = np.stack([self.buf[i][3] for i in idx])
        D = np.array([self.buf[i][4] for i in idx], dtype=np.float32)
        return (torch.from_numpy(S), torch.from_numpy(A), torch.from_numpy(R),
                torch.from_numpy(S2), torch.from_numpy(D))

In [ ]:
def train_dqn(n_episodes=300, gamma=0.99, lr=1e-3, batch=64, target_sync=100,
              warmup=1000, opt_every=4, eps0=1.0, eps_min=0.02, eps_steps=5000, seed=SEED):
    # setup
    set_seed(seed)
    env = gym.make("CartPole-v1")
    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n
    q_net = QNetwork(obs_dim, n_actions)
    target_net = QNetwork(obs_dim, n_actions)
    target_net.load_state_dict(q_net.state_dict())   # the frozen Q^-
    opt = torch.optim.Adam(q_net.parameters(), lr=lr)
    buf = ReplayBuffer(10000)
    returns, losses, eps_log, steps = [], [], [], 0

    # episodes
    for ep in range(n_episodes):
        s, _ = env.reset(seed=seed + ep)
        R, done = 0.0, False
        while not done:
            # decay epsilon
            eps = eps0 + (eps_min - eps0) * min(steps / eps_steps, 1.0)
            eps_log.append(eps)

            # take action using eps-greedy
            if np.random.rand() < eps:
                a = env.action_space.sample()
            else:
                with torch.no_grad():
                    a = q_net(torch.tensor(s, dtype=torch.float32)).argmax().item()

            s_next, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated

            #push onto buffer
            buf.push(s, a, r, s_next, terminated)
            s = s_next
            R += r
            steps += 1

            # gradient step every few steps
            if steps >= warmup and steps % opt_every == 0 and len(buf.buf) >= batch:
                S, A, R_, S2, D = buf.sample(batch)
                q = q_net(S).gather(1, A.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    # the DQN target: r + gamma max_a' Q^-(s', a'), stop-grad, bootstrapped
                    # unless the episode truly terminated
                    target = R_ + gamma * (1 - D) * target_net(S2).max(1).values
                loss = F.mse_loss(q, target)
                opt.zero_grad()
                loss.backward()
                opt.step()
                losses.append(loss.item())

            # stream target network weights every few steps
            if steps % target_sync == 0:
                target_net.load_state_dict(q_net.state_dict())
                
        returns.append(R)
        if (ep + 1) % 25 == 0:
            print(f"episode {ep + 1:3d}: return {R:6.1f}   eps {eps:.2f}")
    env.close()
    return q_net, returns, losses, eps_log

q_net, dqn_returns, dqn_losses, dqn_eps = train_dqn(n_episodes=500)
RESULTS["dqn"] = dqn_returns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].plot(smooth(dqn_returns, 20), color="#4C72B0", lw=2)
axes[0].axhline(500, ls="--", color="k", alpha=0.4, lw=1)
axes[0].set_title("DQN on CartPole-v1 — smoothed return (dashed = solved, 500)")
axes[0].set_xlabel("episode")
axes[1].plot(smooth(dqn_losses, 50), color="#DD8452", lw=2)
axes[1].set_title("TD loss (per optimizer step)")
axes[1].set_xlabel("optimizer step")
axes[2].plot(dqn_eps, color="#55A868", lw=2)
axes[2].set_title("ε over environment steps")
axes[2].set_xlabel("environment step")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def rollout_states(env_id, policy_fn, max_steps=500, seed=SEED):
    set_seed(seed)
    env = gym.make(env_id)
    s, _ = env.reset(seed=seed)
    states, actions, done = [s], [], False
    while not done and len(states) < max_steps:
        a = policy_fn(s)
        actions.append(a)
        s, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        states.append(s)
    env.close()
    return np.array(states), actions


def animate_cartpole(states, actions, title, every=2, fps=25):
    # draw the classic cart-pole figure from the raw state vector
    # (CartPole state = [x, x_dot, theta, theta_dot], theta = 0 is vertical),
    # with an arrow under the cart showing the action taken at each step:
    # action 0 = push left, action 1 = push right
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.set_xlim(-2.6, 2.6)
    ax.set_ylim(-0.4, 1.4)
    ax.axhline(0, color="k", lw=3)                    # track
    ax.plot([-2.4, 2.4], [0.02, 0.02], color="gray", lw=2, ls="--", alpha=0.6)
    ax.set_yticks([])
    ax.set_title(title)
    cart = plt.Rectangle((-0.15, 0.0), 0.3, 0.18, color="#4C72B0")
    pole, = ax.plot([], [], lw=5, color="#C44E52")
    tip, = ax.plot([], [], "o", ms=8, color="#DD8452")
    arrow, = ax.plot([], [], lw=3, ms=10, markevery=[1])   # action arrow under the cart
    label = ax.text(1.45, 1.25, "", ha="left", fontsize=11)
    ax.add_patch(cart)

    def update(i):
        # states[i] is the state before action i, and the final state has no
        # action of its own, so clamp the action lookup one step behind it
        j = min(i, len(actions) - 1)
        x, _, theta, _ = states[i]
        px, py = x + 0.7 * np.sin(theta), 0.18 + 0.7 * np.cos(theta)
        cart.set_x(x - 0.15)
        pole.set_data([x, px], [0.18, py])
        tip.set_data([px], [py])
        a = actions[j]
        if a == 0:  # push left
            arrow.set_data([x - 0.1, x - 0.5], [-0.15, -0.15])
            arrow.set_marker("<")
            arrow.set_color("#4C72B0")
            label.set_text("action: push left")
            label.set_color("#4C72B0")
        else:       # push right
            arrow.set_data([x + 0.1, x + 0.5], [-0.15, -0.15])
            arrow.set_marker(">")
            arrow.set_color("#DD8452")
            label.set_text("action: push right")
            label.set_color("#DD8452")
        return (cart, pole, tip, arrow, label)

    frames = range(0, len(states), every)
    ani = animation.FuncAnimation(fig, update, frames=frames, interval=1000 / fps, blit=True)
    html = ani.to_jshtml()
    plt.close(fig)
    return html


def dqn_policy(s):
    with torch.no_grad():
        return q_net(torch.tensor(s, dtype=torch.float32)).argmax().item()

states, actions = rollout_states("CartPole-v1", dqn_policy, max_steps=500)
print(f"the trained DQN policy balanced for {len(states) - 1} steps")
display(HTML(animate_cartpole(states, actions, "trained DQN on CartPole-v1 (played back 2x speed)")))

**Recap:** the loss only trains $Q_\theta$ — the target is `detach`ed, so gradients flow through $Q_\theta(s_i,a_i)$ only. The target network's parameters are copied from $Q_\theta$ every 100 steps, so the regression target changes slowly and training stays stable. ε decays from 1.0 to 0.02 while the replay buffer fills, then the agent exploits what it learned.

## Policy learning and REINFORCE

- Instead of deriving behavior from values, learn the policy $\pi_\theta(a\mid s)$ directly — a softmax over actions for discrete spaces, a Gaussian density for continuous ones.
- Maximize $J(\theta)=\mathbb E_{a\sim\pi_\theta(a\mid s)}\left[\sum_{k=0}^{\infty}\gamma^k r_{t+k}\right]$.
- Like training a classifier, raise the probability of actions that did well — but weight by *how* well:

$$\nabla_\theta J(\theta)=\mathbb E_{\pi}\Big[\sum_{t=0}^{T}G_t\,\nabla_\theta\log\pi_\theta(a_t\mid s_t)\Big],
\qquad G_t=\sum_{k=0}^{T-t}\gamma^k r_{t+k}$$

- Approximate the expectation with $N$ sampled trajectories (Monte Carlo again!).
- We *maximize* $J$; PyTorch *minimizes*, so we implement `loss = -(G * log_prob).mean()` — descending that loss is ascending $J$. We also standardize the batch returns, a standard variance trick.

In [ ]:
class PolicyNet(nn.Module):
    # discrete actions: softmax over action logits
    def __init__(self, obs_dim, n_actions, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, s):
        return F.softmax(self.net(s), dim=-1)


def collect_episode(policy, env_id, seed, gamma=0.99):
    # one episode -> (states, actions, returns G_t, log pi(a_t|s_t)) for every step
    set_seed(seed)
    env = gym.make(env_id)
    s, _ = env.reset(seed=seed)
    states, actions, rewards, logps = [], [], [], []
    done = False
    while not done:
        states.append(s)
        p = policy(torch.tensor(s, dtype=torch.float32))
        dist = torch.distributions.Categorical(p)
        a = dist.sample()          # the sample is detached; log_prob keeps the graph
        s_next, r, terminated, truncated, _ = env.step(a.item())
        done = terminated or truncated
        actions.append(a.item())
        rewards.append(r)
        logps.append(dist.log_prob(a))
        s = s_next
    env.close()
    G = torch.tensor(compute_returns(np.asarray(rewards, dtype=float), gamma), dtype=torch.float32)
    return np.asarray(states, dtype=np.float32), actions, G, torch.stack(logps)


def train_reinforce(env_id, n_updates=250, n_episodes=10, gamma=0.99, lr=1e-3,
                    seed=SEED, print_every=25):
    set_seed(seed)
    env = gym.make(env_id)
    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
    env.close()
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    returns = []
    for u in range(n_updates):
        logps_batch, G_batch, rets = [], [], []
        for k in range(n_episodes):
            _, _, G, logps = collect_episode(policy, env_id, seed + u * 1000 + k, gamma)
            logps_batch.append(logps)
            G_batch.append(G)
            rets.append(G[0].item())
        logps = torch.cat(logps_batch)
        G = torch.cat(G_batch)
        G = (G - G.mean()) / (G.std() + 1e-8)

        #TODO: calculate the loss we need to backprop.
        #NOTE: Even though RL convention says to maximize objectives, they are computationally still minimized, so you would minimize the negative objective
        loss = None

        
        opt.zero_grad()
        loss.backward()
        opt.step()
        returns.extend(rets)
        if (u + 1) % print_every == 0:
            print(f"update {u + 1:3d}: mean return {np.mean(rets):7.1f}")
    return policy, returns

In [ ]:
rf_policy, rf_returns = train_reinforce("CartPole-v1", n_updates=250, n_episodes=10)
RESULTS["reinforce"] = rf_returns

In [ ]:
def action_prob_episode(policy, env_id, seed, max_steps=500):
    set_seed(seed)
    env = gym.make(env_id)
    s, _ = env.reset(seed=seed)
    probs_left, actions, done, steps = [], [], False, 0
    while not done:
        with torch.no_grad():
            p = policy(torch.tensor(s, dtype=torch.float32))
        a = torch.multinomial(p, 1).item()
        probs_left.append(p[0].item())   # action 0 = push left
        actions.append(a)
        s, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        steps += 1
    env.close()
    return np.array(probs_left), np.array(actions)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.6))
axes[0].plot(smooth(rf_returns, 50), color="#4C72B0", lw=2)
# the curve below is the discounted return G_t (gamma = 0.99), so its ceiling is
# 1 / (1 - gamma) = 100 —, not CartPole's undiscounted 500-step maximum
axes[0].axhline(1 / (1 - 0.99), ls="--", color="k", alpha=0.4)
axes[0].set_title(r"REINFORCE on CartPole-v1 — smoothed discounted return $G_0$" "\n"
                  r"(dashed = ceiling $1/(1-\gamma)=100$)")
axes[0].set_xlabel("episode")
axes[0].set_ylabel(r"$G_0$ (discounted, $\gamma=0.99$)")
axes[0].grid(alpha=0.3)

set_seed(SEED)  # deterministic network initialization
env_check = gym.make("CartPole-v1")
untrained = PolicyNet(env_check.observation_space.shape[0], env_check.action_space.n)
env_check.close()
p_untrained, a_untrained = action_prob_episode(untrained, "CartPole-v1", seed=SEED)
p_trained, a_trained = action_prob_episode(rf_policy, "CartPole-v1", seed=SEED)
axes[1].plot(p_untrained, lw=1.5, color="#DD8452", label="untrained policy")
axes[1].plot(p_trained, lw=1.5, color="#55A868", label="trained policy")
axes[1].set_title(r"$\pi(a=\mathrm{left}\mid s_t)$ during one episode")
axes[1].set_xlabel("step within episode")
axes[1].set_ylabel(r"$p(\mathrm{left}\mid s_t)$")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def reinforce_policy(s):
    with torch.no_grad():
        return rf_policy(torch.tensor(s, dtype=torch.float32)).argmax().item()


rf_states, rf_actions = rollout_states("CartPole-v1", reinforce_policy, max_steps=500)
print(f"the trained REINFORCE policy balanced for {len(rf_states) - 1} steps")
display(HTML(animate_cartpole(rf_states, rf_actions, "trained REINFORCE on CartPole-v1 (played back 2x speed)")))


In [ ]:
class GaussianPolicy(nn.Module):
    # continuous actions: parameterize a Gaussian density over the action
    def __init__(self, obs_dim, hidden=32, log_std_init=-0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )
        self.log_std = nn.Parameter(torch.tensor(log_std_init))

    def forward(self, s):
        mu = self.net(s).squeeze(-1)
        std = self.log_std.exp().clamp(min=1e-3)
        return mu, std

    def sample(self, s):
        # Draw with .sample(), not .rsample(). REINFORCE has to score the action it
        # actually took; rsample() leaves the reparameterisation path attached, and for
        # a Normal that path cancels the score term exactly (d log p / d mu == 0) — the
        # mean would then get no gradient at all and the policy could never improve.
        mu, std = self.forward(s)
        dist = torch.distributions.Normal(mu, std)
        a = dist.sample()
        return a, dist.log_prob(a)


class ContinuousBandit:
    """One step, one continuous action, no state. The reward is -(a - target)^2, so the
    best possible action is exactly `target` and the episode is over immediately. With no
    state and no future there is no credit assignment to do — the policy's only job is to
    decide WHERE to put its Gaussian and how wide to make it, which is precisely what the
    score function below is estimating."""

    def __init__(self, target=1.5):
        self.target = target

    def reset(self):
        return np.zeros(1, dtype=np.float32)      # a constant dummy state

    def step(self, a):
        return self.reset(), -(a - self.target) ** 2


def train_reinforce_cont(env, n_updates=400, n_episodes=16, lr=3e-2,
                         seed=SEED, print_every=100):
    set_seed(seed)
    policy = GaussianPolicy(1)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    returns, means, stds = [], [], []
    with torch.no_grad():
        mu, std = policy(torch.zeros(1))
    means.append(float(mu))          # the untrained policy, so the plot shows the whole
    stds.append(float(std))          # journey rather than only the post-update states
    for u in range(n_updates):
        G_batch, logps_batch, rets = [], [], []
        for k in range(n_episodes):
            set_seed(seed + u * 1000 + k)
            s = env.reset()
            a, logp = policy.sample(torch.tensor(s, dtype=torch.float32))
            _, r = env.step(float(a))
            G_batch.append(torch.tensor([r], dtype=torch.float32))   # 1-step return
            logps_batch.append(logp)
            rets.append(r)
        G = torch.cat(G_batch)
        logps = torch.stack(logps_batch)
        # subtract the batch mean (an unbiased baseline) and rescale to unit variance
        G = (G - G.mean()) / (G.std() + 1e-8)
        opt.zero_grad()
        (-(G * logps).mean()).backward()
        opt.step()
        returns.extend(rets)
        with torch.no_grad():
            mu, std = policy(torch.zeros(1))
        means.append(float(mu))
        stds.append(float(std))
        if (u + 1) % print_every == 0:
            print(f"update {u + 1:3d}: mean action {means[-1]:+.3f}  "
                  f"(target {env.target:+.3f})   std {stds[-1]:.3f}")
    return policy, returns, means, stds

In [ ]:
# A continuous-action bandit: one step, one action, no state, reward -(a - target)^2, so
# the best action is exactly `target` and a perfect score is 0. There is no episode and
# nothing to assign credit to, which strips REINFORCE down to its bare mechanism: the
# score function nudges mu toward the actions that paid off and shrinks sigma where the
# policy is already confident. (CartPole above had to solve a harder problem — working out
# WHICH state deserved the credit — and that is exactly what the next section adds a
# critic for.)
bandit = ContinuousBandit(target=1.5)
band_policy, band_returns, band_means, band_stds = train_reinforce_cont(
    bandit, n_updates=400, n_episodes=16, lr=3e-2)
RESULTS["bandit"] = band_returns

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))

# left: where the policy puts its action, at a few points in training
a_grid = np.linspace(-2.5, 4.5, 400)
axes[0].plot(a_grid, -(a_grid - bandit.target) ** 2, color="gray", lw=1.5,
             label=r"reward $-(a-1.5)^2$")
for u, color in [(0, "#C44E52"), (25, "#DD8452"), (100, "#55A868"), (400, "#4C72B0")]:
    m, sd = band_means[u], band_stds[u]
    axes[0].axvspan(m - sd, m + sd, color=color, alpha=0.15)
    label = "untrained" if u == 0 else f"update {u}"
    axes[0].axvline(m, color=color, lw=2, label=f"{label}: mu {m:+.2f}, sd {sd:.2f}")
axes[0].axvline(bandit.target, ls="--", color="k", alpha=0.5)
axes[0].set_xlabel("action a")
axes[0].set_ylabel("reward")
axes[0].set_title("the policy migrates to the peak of the reward and narrows")
axes[0].legend(fontsize=8, loc="lower left")
axes[0].grid(alpha=0.3)

# right: the mean action over training
axes[1].plot(band_means, color="#4C72B0", lw=2)
axes[1].axhline(bandit.target, ls="--", color="k", alpha=0.5)
axes[1].set_xlabel("update")
axes[1].set_ylabel(r"policy mean $\mu$ (the deterministic action)")
axes[1].set_title(r"$\mu$ converges to the optimum $a^\star = 1.5$")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
class ValueNet(nn.Module):
    # the baseline: V_phi(s)
    def __init__(self, obs_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, s):
        return self.net(s).squeeze(-1)


def train_reinforce_baseline(env_id, n_updates=200, n_episodes=10, gamma=0.99,
                             lr_actor=1e-3, lr_critic=1e-3, seed=SEED, print_every=25):
    set_seed(seed)
    env = gym.make(env_id)
    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
    baseline = ValueNet(env.observation_space.shape[0])
    env.close()
    opt_p = torch.optim.Adam(policy.parameters(), lr=lr_actor)
    opt_v = torch.optim.Adam(baseline.parameters(), lr=lr_critic)
    returns = []
    for u in range(n_updates):
        S, logps, G = [], [], []
        rets = []
        for k in range(n_episodes):
            states, _, G_k, logps_k = collect_episode(policy, env_id, seed + u * 1000 + k, gamma)
            S.append(states)
            G.append(G_k)
            logps.append(logps_k)
            rets.append(G_k[0].item())
        S_t = torch.tensor(np.concatenate(S), dtype=torch.float32)
        logps = torch.cat(logps)
        G = torch.cat(G)
        V = baseline(S_t)
        # the advantage A_t = G_t - V_phi(s_t); the baseline is detached
        advantage = G - V.detach()
        advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-8)
        actor_loss = -(advantage * logps).mean()
        critic_loss = F.mse_loss(V, G)   # the baseline learns to predict the return
        opt_p.zero_grad()
        actor_loss.backward()
        opt_p.step()
        opt_v.zero_grad()
        critic_loss.backward()
        opt_v.step()
        returns.extend(rets)
        if (u + 1) % print_every == 0:
            print(f"update {u + 1:3d}: mean return {np.mean(rets):7.1f}")
    return policy, returns

In [ ]:
_, rf_base_returns = train_reinforce_baseline("CartPole-v1", n_updates=200, n_episodes=10)
RESULTS["reinforce_base"] = rf_base_returns

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(smooth(rf_returns, 50), lw=2, color="#DD8452", label="REINFORCE (no baseline)")
ax.plot(smooth(rf_base_returns, 50), lw=2, color="#55A868", label="REINFORCE + value baseline")
# both curves are discounted G_t (gamma = 0.99), so the ceiling is 1/(1-gamma) = 100
ax.axhline(1 / (1 - 0.99), ls="--", color="k", alpha=0.4)
ax.set_xlabel("episode")
ax.set_ylabel(r"$G_0$ (discounted, smoothed)")
ax.set_title(r"A learned baseline reduces variance" "\n"
             r"(dashed = ceiling $1/(1-\gamma)=100$)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## From REINFORCE to actor-critic

- Plain REINFORCE has three weaknesses: $G_t$ is noisy (high variance), there is no **credit assignment** (which action earned the return?), and it must wait for the episode to end.
- Subtract a learned baseline — the **advantage** $A_t=G_t-V_\phi(s_t)$: *"better or worse than expected from here?"*
- Now bring TD back: $A_t^{\text{actor-critic}}=\delta_t=r_t+\gamma V_\phi(s_{t+1})-V_\phi(s_t)$ — available after **one step**, so both models can be updated *every* timestep.
- **Actor (gradient ascent):** $\theta\leftarrow\theta+\alpha\,\operatorname{sg}[\delta_t]\,\nabla_\theta\log\pi_\theta(a_t\mid s_t)$.
- **Critic (gradient descent):** $\phi\leftarrow\phi-\beta\,\nabla_\phi\,\frac12\big(V_\phi(s_t)-\operatorname{sg}[r_t+\gamma V_\phi(s_{t+1})]\big)^2$.

The actor **maximizes** an objective; the critic **minimizes** a loss — hence ascent vs. descent. In PyTorch both become `loss`es: the actor's is $-\operatorname{sg}[\delta_t]\log\pi$, and the critic's is $\frac12\delta_t^2$.

In [ ]:
class ActorNet(nn.Module):
    # the actor: pi_theta(a|s), softmax for discrete actions
    def __init__(self, obs_dim, n_actions, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, s):
        return F.softmax(self.net(s), dim=-1)


class CriticNet(nn.Module):
    # the critic: V_phi(s)
    def __init__(self, obs_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, s):
        return self.net(s).squeeze(-1)


def train_a2c(env_id, n_episodes=1500, gamma=0.99, lr_actor=3e-4, lr_critic=1e-3,
              seed=SEED, print_every=100):
    set_seed(seed)
    env = gym.make(env_id)
    actor = ActorNet(env.observation_space.shape[0], env.action_space.n)
    critic = CriticNet(env.observation_space.shape[0])
    opt_a = torch.optim.Adam(actor.parameters(), lr=lr_actor)
    opt_v = torch.optim.Adam(critic.parameters(), lr=lr_critic)
    returns, delta_log = [], []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=seed + ep)
        ep_return, done = 0.0, False
        deltas = []
        while not done:
            p = actor(torch.tensor(s, dtype=torch.float32))
            dist = torch.distributions.Categorical(p)
            a = dist.sample()
            s_next, r, terminated, truncated, _ = env.step(a.item())
            done = terminated or truncated
            V = critic(torch.tensor(s, dtype=torch.float32))
            with torch.no_grad():
                V2 = critic(torch.tensor(s_next, dtype=torch.float32))
            # the one-step TD error: delta_t = r_t + gamma V(s_{t+1}) - V(s_t), bootstrapped
            # unless the episode truly terminated
            delta = r + gamma * (1 - float(terminated)) * V2 - V
            # actor ascent  <=>  descend -sg[delta_t] log pi(a_t|s_t)
            actor_loss = -(delta.detach() * dist.log_prob(a))
            # critic descent on 1/2 (V - sg[target])^2
            critic_loss = 0.5 * delta.pow(2)
            opt_a.zero_grad()
            actor_loss.backward()
            opt_a.step()
            opt_v.zero_grad()
            critic_loss.backward()
            opt_v.step()
            s = s_next
            ep_return += r
            deltas.append(delta.abs().item())
        returns.append(ep_return)
        delta_log.append(np.mean(deltas))
        if (ep + 1) % print_every == 0:
            print(f"episode {ep + 1:4d}: return {ep_return:7.1f}   mean |delta_t| {delta_log[-1]:.2f}")
    env.close()
    return actor, critic, returns, delta_log

In [ ]:
# Acrobot-v1: two linked pendulum arms, 3 actions (torque −1/0/+1), reward −1 per step.
# The agent must swing the tip above the goal line; gymnasium calls it "solved"
# at an average return of −100 (dashed line).
a2c_actor, a2c_critic, a2c_returns, a2c_delta = train_a2c("Acrobot-v1", n_episodes=1500)
RESULTS["a2c"] = a2c_returns

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.6))
axes[0].plot(smooth(a2c_returns, 50), color="#4C72B0", lw=2)
axes[0].axhline(-100, ls="--", color="k", alpha=0.4)
axes[0].set_title("one-step actor-critic on Acrobot-v1")
axes[0].set_xlabel("episode")
axes[0].set_ylabel("return (smoothed)")
axes[0].grid(alpha=0.3)
axes[1].plot(smooth(a2c_delta, 50), color="#DD8452", lw=2)
axes[1].set_title(r"mean $|\delta_t|$ per episode — the critic's one-step error shrinks")
axes[1].set_xlabel("episode")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Watch the trained actor-critic agent swing the acrobot up to the goal line.
# (acrobot state = [cos th1, sin th1, cos th2, sin th2, th1_dot, th2_dot];
#  th1 = 0 hangs straight down; torque actions: 0 = -1, 1 = 0, 2 = +1)
def acrobot_policy(s):
    with torch.no_grad():
        return a2c_actor(torch.tensor(s, dtype=torch.float32)).argmax().item()


def animate_acrobot(states, actions, title, every=2, fps=25):
    fig, ax = plt.subplots(figsize=(5.5, 4.6))
    ax.set_xlim(-2.2, 2.2)
    ax.set_ylim(-2.2, 1.35)
    ax.set_aspect("equal")
    ax.axhline(1.0, color="green", ls="--", lw=2, alpha=0.7)   # goal line
    ax.text(2.05, 1.03, "GOAL", ha="right", va="bottom", fontsize=9, color="green")
    ax.plot([0], [0], "o", ms=8, color="k")                    # pivot
    link1, = ax.plot([], [], lw=6, color="#4C72B0", solid_capstyle="round")
    link2, = ax.plot([], [], lw=6, color="#DD8452", solid_capstyle="round")
    joint, = ax.plot([], [], "o", ms=10, color="#C44E52")
    label = ax.text(-2.05, 1.25, "", ha="left", fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)

    def update(i):
        # states[i] is the state before action i, and the final state has no
        # action of its own, so clamp the action lookup one step behind it
        j = min(i, len(actions) - 1)
        c1, s1, c2, s2 = states[i][:4]
        th1 = np.arctan2(s1, c1)
        th2 = np.arctan2(s2, c2)
        jx, jy = np.sin(th1), -np.cos(th1)
        tx, ty = jx + np.sin(th1 + th2), jy - np.cos(th1 + th2)
        link1.set_data([0, jx], [0, jy])
        link2.set_data([jx, tx], [jy, ty])
        joint.set_data([jx], [jy])
        torque = [-1, 0, 1][actions[j]]
        color = "#4C72B0" if torque < 0 else "#DD8452" if torque > 0 else "gray"
        label.set_text(f"action: torque {torque:+d}")
        label.set_color(color)
        return (link1, link2, joint, label)

    frames = range(0, len(states), every)
    ani = animation.FuncAnimation(fig, update, frames=frames, interval=1000 / fps, blit=True)
    html = ani.to_jshtml()
    plt.close(fig)
    return html

ac_states, ac_actions = rollout_states("Acrobot-v1", acrobot_policy, max_steps=500)
print(f"the trained actor-critic swung the acrobot up in {len(ac_states) - 1} steps")
display(HTML(animate_acrobot(ac_states, ac_actions, "trained actor-critic on Acrobot-v1 (played back 2x speed)")))

In [ ]:
# What did the critic actually learn? V_phi(s) over the acrobot's two joint angles
# (acrobot state = [cos th1, sin th1, cos th2, sin th2, th1_dot, th2_dot]; velocities = 0 here).
# Bright = the critic thinks the tip is close to clearing the goal line.
n1 = n2 = 50
g1 = np.linspace(-np.pi, np.pi, n1)
g2 = np.linspace(-np.pi, np.pi, n2)
grid = np.zeros((n1 * n2, 6), dtype=np.float32)
for i, th2 in enumerate(g2):
    for j, th1 in enumerate(g1):
        grid[i * n1 + j] = [np.cos(th1), np.sin(th1), np.cos(th2), np.sin(th2), 0.0, 0.0]
with torch.no_grad():
    Vg = a2c_critic(torch.tensor(grid)).numpy().reshape(n2, n1)

fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(Vg, extent=[-np.pi, np.pi, -np.pi, np.pi], origin="lower", aspect="auto", cmap="viridis")
ax.set_xlabel("θ₁ (rad)")
ax.set_ylabel("θ₂ (rad)")
ax.set_title(r"learned critic $V_\phi(\theta_1, \theta_2)$ with both velocities 0")
plt.colorbar(im, ax=ax)
plt.show()

## Thanks!

Ideas to play with next: swap environments, tune $\gamma$ and $\varepsilon$, try `Acrobot-v1` with A2C, or re-run a section with a different seed to see the variance RL keeps warning about.